# Exp-08 YOLO Object Detection 

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load YOLO model
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

# Load class names
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Get output layer names
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

# Load image
img = cv2.imread("fox.png")
height, width, _ = img.shape

# ---------------------------
# Prepare image for YOLO
# ---------------------------
blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416),
                             swapRB=True, crop=False)

net.setInput(blob)
outputs = net.forward(output_layers)

# ---------------------------
# Process detections
# ---------------------------
boxes = []
confidences = []
class_ids = []

for output in outputs:
    for detection in output:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if confidence > 0.5:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

# ---------------------------
# Remove duplicates (NMS)
# ---------------------------
indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

# Draw results
for i in range(len(boxes)):
    if i in indexes:
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# ---------------------------
# Show result (Jupyter)
# ---------------------------
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title("YOLO Detection")
plt.axis('off')
plt.show()